# معمل التجارب: التقطيع الدلالي الجاهز باستخدام LangChain (المحاضرة 40)
## القسم الأول: نظرة عامة وتوجيهات المعمل

في هذا المعمل العملي:
1. ستتعلم وتطبق استخدام أداة `SemanticChunker` الرسمية المدمجة في LangChain.
2. ستختبر استراتيجيات العتبات الإحصائية المختلفة (`percentile`, `standard_deviation`, `interquartile`).
3. ستطبق تكليف المحاضرة ببناء مسار RAG كامل باستخدام `FAISS` وموديل `Groq LLM` وسلسلة `LCEL`.


## القسم الثاني: الكود المرجعي للتطبيق والتجربة
شغل الخلايا التالية لمتابعة عمل الخوارزمية، ثم استخدم مساحة التجربة الحرة في الأسفل للتعديل.

In [1]:
# 1. استدعاء المكتبات وإعداد المستند
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

# LangChain libraries 
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker


# Vector store and retrieval
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [2]:
# load the document
loader = TextLoader("../../stages/04-advanced-chunking/04-semantic-chunking-langchain/langchain_intro.txt")
docs = loader.load()

# embedings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# create a semantic chunker
chunker = SemanticChunker(embeddings=embeddings)

# split the document into semantic chunks
chunks = chunker.split_documents(docs)

# print the number of chunk
print(f"[INFO] Document loaded. Characters: {len(docs[0].page_content   )}")
print(f"[INFO] Number of chunks: {len(chunks)}")
print("-"*50)
print(f"[INFO] Total chunks produced: {len(chunks)}\n")
for idx, c in enumerate(chunks, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{c.page_content}\n")




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Document loaded. Characters: 287
[INFO] Number of chunks: 2
--------------------------------------------------
[INFO] Total chunks produced: 2

--- Chunk [1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

--- Chunk [2] ---
The Eiffel Tower is located in Paris. France is a popular tourist destination.



In [3]:
# 2. تشغيل SemanticChunker الافتراضي
chunker = SemanticChunker(embeddings)
chunks = chunker.split_documents(docs)

print(f"[INFO] Total chunks produced: {len(chunks)}\n")
for idx, c in enumerate(chunks, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{c.page_content}\n")

[INFO] Total chunks produced: 2

--- Chunk [1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

--- Chunk [2] ---
The Eiffel Tower is located in Paris. France is a popular tourist destination.



In [4]:
# 3. تكليف المحاضرة: بناء مسار RAG كامل مع FAISS و Groq

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_template("""Answer based only on the context:
Context: {context}
Question: {question}
Answer:""")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)
question = "What tools can LangChain be combined with?"
response = rag_chain.invoke({"question": question})

print(f"[QUESTION]: {question}")
print(f"[RESPONSE]: {response}")
print("-"*50)
print(f"Chunk used for the answer: {retriever.invoke(question)[0].page_content}")

[QUESTION]: What tools can LangChain be combined with?
[RESPONSE]: LangChain can be combined with tools such as **OpenAI** and **Pinecone**.
--------------------------------------------------
Chunk used for the answer: LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.
